In [ ]:
import ee
import geemap
ee.Authenticate()
ee.Initialize(project='geo-data-s')
print("✅ Earth Engine iniciado com sucesso.")
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


✅ Earth Engine iniciado com sucesso.
Mounted at /content/drive


In [ ]:
naturais = [3, 4, 12]
antropicas = [21, 22, 29]
anos = list(range(1985, 2025, 5))
limiares = [0.2, 0.4, 0.6, 0.8]

classeDictNaturais = {
    'FORMAÇÃO FLORESTAL': 3,
    'FORMAÇÃO SAVÂNICA': 4,
    'FORMAÇÃO CAMPESTRE': 12
}

classeDictAntropicas = {
 'PASTAGEM': 21, 'AGRICULTURA': 21, 'LAVOURA TEMPORÁRIA': 21, 'SOJA': 21,
  'CANA': 21, 'ARROZ': 21, 'ALGODÃO': 21, 'OUTRAS LAVOURAS TEMPORÁRIAS': 21,
  'LAVOURA PERENE': 21, 'CAFÉ': 21, 'CITRUS': 21, 'DENDÊ': 21,
  'OUTRAS LAVOURAS PERENES': 21,  'MOSAICO DE USOS': 21,
  'PRAIA, DUNA E AREAL': 22, 'PRAIA E DUNA': 22, 'ÁREA URBANIZADA': 22,
  'VEGETAÇÃO URBANA': 22, 'INFRAESTRUTURA URBANA': 22, 'MINERAÇÃO': 22,
  'OUTRAS ÁREAS NÃO VEGETADAS': 22, 'OUTRA ÁREA NÃO VEGETADA': 22,
  'APICUM': 22, 'AFLORAMENTO ROCHOSO': 29
}
asset_bacias = 'projects/mapbiomas-workspace/AMOSTRAS/col9/CAATINGA/bacias_hidrografica_caatinga_49_regions'
bacias = ee.FeatureCollection(asset_bacias)


In [ ]:
def avaliar_bacia(idBacia):
    print(f"🟡 Avaliando bacia {idBacia}...")
    asset_filtros = 'projects/mapbiomas-workspace/AMOSTRAS/col10/CAATINGA/POS-CLASS/Spatials_int'
    img = (ee.ImageCollection(asset_filtros)
                .filter(ee.Filter.eq('version', 10))
                .filter(ee.Filter.eq('id_bacias', idBacia))
            )
    numeroImg = img.size().getInfo()
    print(f"📦 Tamanho da coleção para bacia {idBacia}: {numeroImg}")

    if numeroImg  > 0:
        raise Exception(f"❌ A coleção está vazia para bacia {idBacia}")

    if 'Temporal' in asset_filtros:
        img = img.filter(ee.Filter.eq('janela', 5)).min()
    else:
        img = img.min()


    proj = img.select('classification_2015').projection()
    geom = bacias.filter(ee.Filter.eq('nunivotto4', idBacia)).geometry()
    asset_points = 'projects/mapbiomas-workspace/VALIDACAO/mapbiomas_85k_col4_points_w_edge_and_edited_v1'
    pontos = ee.FeatureCollection(asset_points).filterBounds(geom)

    def avaliar_combo(classes, classe_dict, label, ano):
        classe_keys = list(classe_dict.keys())
        dict_js = ee.Dictionary(classe_dict)

        pontos_valid = pontos\
            .filter(ee.Filter.inList(f'CLASS_{ano}', classe_keys))\
            .map(lambda f: f.set('ref', dict_js.get(f.get(f'CLASS_{ano}'))))
        # Lista de bandas disponíveis no asset
        bandas_disponiveis = img.bandNames()

        # Gera lista de bandas válidas disponíveis
        bandas = ['classification_' + str(a) for a in range(1985, 2025)]
        bandas_validas = [b for b in bandas if bandas_disponiveis.contains(b).getInfo()]

        stack = ee.ImageCollection([
            img.select(b).updateMask(img.select(b).remap(classes, [1]*len(classes)).eq(1)).rename(b)
            for b in bandas_validas
        ]).toBands()

        freqs = {c: stack.eq(c).reduce(ee.Reducer.sum()) for c in classes}
        freqTot = stack.neq(0).reduce(ee.Reducer.count())

        combinacoes = [[l1, l2, l3] for l1 in limiares for l2 in limiares for l3 in limiares]
        melhores = {'acc': -1, 'l1': None, 'l2': None, 'l3': None}

        for lim in combinacoes:
            l1, l2, l3 = lim
            m1 = freqs[classes[0]].divide(freqTot).gte(l1)
            m2 = freqs[classes[1]].divide(freqTot).gte(l2)
            m3 = freqs[classes[2]].divide(freqTot).gte(l3)

            novaClasse = ee.Image(0).where(m1, classes[0])\
                .where(m2.And(m1.Not()), classes[1])\
                .where(m3.And(m1.Not()).And(m2.Not()), classes[2])

            bandaAno = img.select(f'classification_{ano}')
            mascara = bandaAno.remap(classes, [1]*len(classes)).eq(1)

            reclass = bandaAno.where(mascara, novaClasse.unmask(0))\
                .rename('map_class').setDefaultProjection(proj)

            pontos_classificados = pontos_valid.map(lambda f: f.set(
                'map_class',
                reclass.reduceRegion(
                    reducer=ee.Reducer.first(),
                    geometry=f.geometry(),#.centroid(30),
                    scale=30,
                    maxPixels=1e9
                ).get('map_class')
            ))

            pontos_validos = pontos_classificados.filter(ee.Filter.notNull(['map_class', 'ref']))


            pontos_acerto = pontos_validos.map(lambda f: f.set(
                'acerto',
                ee.Number(f.get('ref')).eq(ee.Number(f.get('map_class'))).int()
            ))

            total = pontos_acerto.size()
            acertos = pontos_acerto.aggregate_sum('acerto')
            acc = ee.Number(acertos).divide(total).getInfo()

            print(f"   🔍 {label} — ACC: {acc:.2f} | Limiar: {lim}")
            if acc > melhores['acc']:
                melhores.update({'acc': acc, 'l1': l1, 'l2': l2, 'l3': l3})

        print(f"✅ Melhor {label}: {melhores}")
        return melhores

    melhores_nat = avaliar_combo(naturais, classeDictNaturais, 'Naturais', 2015)
    melhores_ant = avaliar_combo(antropicas, classeDictAntropicas, 'Antrópicas', 2015)
    return {'bacia': idBacia, 'naturais': melhores_nat, 'antropicas': melhores_ant}

In [ ]:
# 2. Bibliotecas e caminho do arquivo
import os
import pandas as pd

caminho_csv = '/content/drive/MyDrive/mapbiomas/featureSelection/resultados_bacias_3.csv'

# Verifica se já existe o CSV para recuperar bacias processadas
if os.path.exists(caminho_csv):
    df_existente = pd.read_csv(caminho_csv)
    bacias_processadas = set(df_existente['bacia'].astype(int).tolist())
else:
    # Cria CSV com cabeçalho se não existir
    colunas = [
        'bacia',
        'acc_naturais', 'l1_naturais', 'l2_naturais', 'l3_naturais',
        'acc_antropicas', 'l1_antropicas', 'l2_antropicas', 'l3_antropicas'
    ]
    df_vazio = pd.DataFrame(columns=colunas)
    df_vazio.to_csv(caminho_csv, index=False)
    bacias_processadas = set()

In [ ]:
resultados = []
# 1. Carrega bacias já processadas, se o arquivo existir
if os.path.exists(caminho_csv):
    df_existente = pd.read_csv(caminho_csv)
    bacias_processadas = set(df_existente['bacia'].astype(int).tolist())

    # Preenche a lista 'resultados' com dados do CSV
    for _, row in df_existente.iterrows():
          resultados.append({
              'bacia': str(int(row['bacia'])),
              'naturais': {
                  'acc': float(row['acc_naturais']),
                  'l1': float(row['l1_naturais']),
                  'l2': float(row['l2_naturais']),
                  'l3': float(row['l3_naturais'])
              },
              'antropicas': {
                  'acc': float(row['acc_antropicas']),
                  'l1': float(row['l1_antropicas']),
                  'l2': float(row['l2_antropicas']),
                  'l3': float(row['l3_antropicas'])
              }
          })

    print(f"🔁 {len(bacias_processadas)} bacias já processadas.")
else:
    # Se não existir, inicializa lista vazia e cria o CSV com cabeçalho
    bacias_processadas = set()
    colunas = [
        'bacia',
        'acc_naturais', 'l1_naturais', 'l2_naturais', 'l3_naturais',
        'acc_antropicas', 'l1_antropicas', 'l2_antropicas', 'l3_antropicas'
    ]
    pd.DataFrame(columns=colunas).to_csv(caminho_csv, index=False)
    print("📄 Arquivo CSV criado com cabeçalho.")

# 2. Lista de todas as bacias da coleção
lista_bacias = bacias.aggregate_array('nunivotto4').distinct().getInfo()
print(f"lista com as {len(lista_bacias)} bacias \n ", lista_bacias)


🔁 49 bacias já processadas.
lista com as 49 bacias 
  ['7411', '7422', '7424', '7438', '7443', '745', '746', '751', '752', '753', '7541', '7544', '755', '7561', '7564', '757', '7581', '7584', '7591', '7592', '761111', '761112', '76116', '7612', '7613', '7614', '7615', '7616', '7617', '7618', '7619', '7622', '7625', '763', '764', '765', '766', '7671', '7691', '771', '7712', '772', '7721', '773', '7741', '7746', '7754', '7761', '7764']


In [ ]:
# 3. Loop principal de processamento
lista_bacias = [ "7613","7746","7741","7591","7581","757"]
for nbacia in lista_bacias:
    if int(b) in bacias_processadas:
        print(f"⏩ Bacia {nbacia} já processada. Pulando...")
        continue

    try:
        res = avaliar_bacia(nbacia)
        resultados.append(res)

        linha = {
            'bacia': res['bacia'],
            'acc_naturais': res['naturais']['acc'],
            'l1_naturais': res['naturais']['l1'],
            'l2_naturais': res['naturais']['l2'],
            'l3_naturais': res['naturais']['l3'],
            'acc_antropicas': res['antropicas']['acc'],
            'l1_antropicas': res['antropicas']['l1'],
            'l2_antropicas': res['antropicas']['l2'],
            'l3_antropicas': res['antropicas']['l3']
        }

        df_linha = pd.DataFrame([linha])
        df_linha.to_csv(caminho_csv, mode='a', header=False, index=False)

        print(f"✅ Bacia {b} processada e salva.")
    except Exception as e:
        print(f"❌ Erro na bacia {b}: {e}")

print("🏁 Avaliação finalizada.")

⏩ Bacia 7613 já processada. Pulando...
⏩ Bacia 7746 já processada. Pulando...
⏩ Bacia 7741 já processada. Pulando...
⏩ Bacia 7591 já processada. Pulando...
⏩ Bacia 7581 já processada. Pulando...
⏩ Bacia 757 já processada. Pulando...
🏁 Avaliação finalizada.


In [ ]:
print(resultados[0])

{'bacia': '7411', 'naturais': {'acc': 0.790625, 'l1': 0.4, 'l2': 0.2, 'l3': 0.2}, 'antropicas': {'acc': 0.4766355140186916, 'l1': 0.2, 'l2': 0.2, 'l3': 0.2}}


In [ ]:
# asset_input = 'projects/mapbiomas-workspace/AMOSTRAS/col10/CAATINGA/POS-CLASS/Spatials'
# colecao = (ee.ImageCollection(asset_input)
#                 .filter(ee.Filter.eq('version', 10))
#                 .filter(ee.Filter.eq('id_bacias', '757')))

# size = colecao.size().getInfo()
# print(size)

1


In [ ]:
def aplicar_filtro_frequencia(idBacia, melhores_nat, melhores_ant):
    print(f"\n⚙️ Aplicando filtro na série temporal da bacia {idBacia}...")

    # 1. Carrega a coleção filtrada por bacia e janela
    asset_input = 'projects/mapbiomas-workspace/AMOSTRAS/col10/CAATINGA/POS-CLASS/Spatials_int'
    colecao = (ee.ImageCollection(asset_input)
                    .filter(ee.Filter.eq('version', 10))
                    .filter(ee.Filter.eq('id_bacias', str(idBacia))))

    size = colecao.size().getInfo()
    print(f"📦 Tamanho da coleção para bacia {idBacia}: {size}")

    if size == 0:
        raise Exception(f"❌ A coleção está vazia para bacia {idBacia}")

    img = colecao.min()

    # 2. Verifica as bandas disponíveis
    try:
        bandas = img.bandNames().getInfo()
        print(f"🔍 Bandas da imagem da bacia {idBacia}: {bandas}")
    except Exception as e:
        raise Exception(f"❌ Erro ao obter bandas da imagem da bacia {idBacia}: {e}")

    # 3. Garante que há uma banda classification_ para pegar projeção
    bandas_validas = [b for b in bandas if b.startswith('classification_')]
    if len(bandas_validas) == 0:
        raise Exception(f"❌ Nenhuma banda 'classification_' encontrada para bacia {idBacia}")

    proj = img.select(bandas_validas[0]).projection()

    # 4. Lista de anos
    anos = list(range(1985, 2025))

    # 5. Função para aplicar máscara por frequência de ocorrência
    def get_freq_mask(classes, limiares):
        stack = ee.ImageCollection([
            img.select(f'classification_{ano}')\
               .updateMask(img.select(f'classification_{ano}').remap(classes, [1]*len(classes)).eq(1))\
               .rename(f'classification_{ano}')
            for ano in anos
        ]).toBands()

        freqs = {c: stack.eq(c).reduce(ee.Reducer.sum()) for c in classes}
        freqTot = stack.neq(0).reduce(ee.Reducer.count())

        l1, l2, l3 = float(limiares['l1']), float(limiares['l2']), float(limiares['l3'])

        m1 = freqs[classes[0]].divide(freqTot).gte(l1)
        m2 = freqs[classes[1]].divide(freqTot).gte(l2)
        m3 = freqs[classes[2]].divide(freqTot).gte(l3)

        novaClasse = ee.Image(0)\
            .where(m1, classes[0])\
            .where(m2.And(m1.Not()), classes[1])\
            .where(m3.And(m1.Not()).And(m2.Not()), classes[2])

        return novaClasse

    # Classes definidas externamente ou definidas aqui
    naturais = [3, 4, 12]
    antropicas = [21, 22, 29]

    mask_nat = get_freq_mask(naturais, melhores_nat)
    mask_ant = get_freq_mask(antropicas, melhores_ant)

    # 6. Função para corrigir cada ano aplicando as máscaras
    def corrigir_ano(ano):
        banda = img.select(f'classification_{ano}')
        mask_n = banda.remap(naturais, [1]*len(naturais)).eq(1)
        mask_a = banda.remap(antropicas, [1]*len(antropicas)).eq(1)

        corrigido = banda\
            .where(mask_n, mask_nat.unmask(banda))\
            .where(mask_a, mask_ant.unmask(banda))\
            .rename(f'classification_{ano}')\
            .setDefaultProjection(proj)
        return corrigido

    imagens_corrigidas = [corrigir_ano(ano) for ano in anos]
    colecao_corrigida = ee.Image.cat(imagens_corrigidas)

    print(f"✅ Filtro aplicado com sucesso para a bacia {idBacia}.")
    return colecao_corrigida

In [ ]:
# --- APLICAÇÃO DO FILTRO E EXPORTAÇÃO POR BACIA ---
for res in resultados:
    try:
        bacia_id = res['bacia']
        melhores_nat = res['naturais']
        melhores_ant = res['antropicas']
        geomet = bacias.filter(ee.Filter.eq('nunivotto4', bacia_id)).geometry()
        # Aplica o filtro de frequência com os melhores limiares
        colecao_corrigida = aplicar_filtro_frequencia(bacia_id, melhores_nat, melhores_ant)
        colecao_corrigida = (colecao_corrigida#.select(self.lstbandNames)
                        #.updateMask(self.bacia_raster)
                        .set(
                            'version', 10,
                            'biome', 'CAATINGA',
                            'type_filter', 'frequence',
                            'from', 'TemporalCC',
                            'collection', '10.0',
                            'model', "GTB",
                            'id_bacias', bacia_id,
                            'sensor', 'Landsat',
                            'system:footprint', geomet )
                    )
        #print(colecao_corrigida.propertyNames().getInfo())
        # Define um nome único para exportação
        asset_output = 'projects/mapbiomas-workspace/AMOSTRAS/col10/CAATINGA/POS-CLASS/Frequency'
        nome_asset = f"filterFQ_BACIA_{bacia_id}_GTB_V10" # <- ajuste o caminho do seu asset
        optExp = {
            'image': colecao_corrigida,
            'description': nome_asset,
            'assetId': os.path.join(asset_output, nome_asset),
            'region': geomet,
            'scale': 30,
            'maxPixels': 1e13,
            "pyramidingPolicy":{".default": "mode"}
        }

        # Exporta para o seu espaço no Earth Engine
        task = ee.batch.Export.image.toAsset(**optExp)
        task.start()
        print(f"📤 Exportação iniciada para a bacia {bacia_id}.")
    except Exception as e:
        print(f"❌ Erro ao exportar bacia {bacia_id}: {e}")


⚙️ Aplicando filtro na série temporal da bacia 7411...
📦 Tamanho da coleção para bacia 7411: 0
❌ Erro ao exportar bacia 7411: ❌ A coleção está vazia para bacia 7411

⚙️ Aplicando filtro na série temporal da bacia 7422...
📦 Tamanho da coleção para bacia 7422: 0
❌ Erro ao exportar bacia 7422: ❌ A coleção está vazia para bacia 7422

⚙️ Aplicando filtro na série temporal da bacia 7424...
📦 Tamanho da coleção para bacia 7424: 0
❌ Erro ao exportar bacia 7424: ❌ A coleção está vazia para bacia 7424

⚙️ Aplicando filtro na série temporal da bacia 7438...
📦 Tamanho da coleção para bacia 7438: 0
❌ Erro ao exportar bacia 7438: ❌ A coleção está vazia para bacia 7438

⚙️ Aplicando filtro na série temporal da bacia 7443...
📦 Tamanho da coleção para bacia 7443: 0
❌ Erro ao exportar bacia 7443: ❌ A coleção está vazia para bacia 7443

⚙️ Aplicando filtro na série temporal da bacia 745...
📦 Tamanho da coleção para bacia 745: 0
❌ Erro ao exportar bacia 745: ❌ A coleção está vazia para bacia 745

⚙️ Apli

In [ ]:

# --- VISUALIZAÇÃO DOS MAPAS CORRIGIDOS COM GEEMAP ---

import geemap
import matplotlib.pyplot as plt

img = ee.ImageCollection('projects/mapbiomas-workspace/AMOSTRAS/col10/CAATINGA/POS-CLASS/TemporalCC')\
        .filter(ee.Filter.eq('version', 7))\
        .filter(ee.Filter.eq('id_bacias', '751')).min()

# --- LISTA DE ANOS PARA VISUALIZAR ---
anos_visualizar = list(range(1985, 2025, 5))  # de 5 em 5 anos
Map = geemap.Map(center=[-10, -40], zoom=6, height=1200)

# --- APLICA O FILTRO E GERA A COLEÇÃO CORRIGIDA ---
colecao_corrigida = aplicar_filtro_frequencia('751', resultados[0]['naturais'], resultados[0]['antropicas'])

# --- PALETA MAPBIOMAS ---
palette = [
    "#ffffff","#32a65e","#32a65e","#1f8d49","#7dc975","#04381d","#026975","#000000","#000000","#7a6c00","#ad975a",
    "#519799","#d6bc74","#d89f5c","#FFFFB2","#edde8e","#000000","#000000","#f5b3c8","#C27BA0","#db7093","#ffefc3",
    "#db4d4f","#ffa07a","#d4271e","#db4d4f","#0000FF","#000000","#000000","#ffaa5f","#9c0027","#091077","#fc8114",
    "#2532e4","#93dfe6","#9065d0","#d082de","#000000","#000000","#f5b3c8","#c71585","#f54ca9","#cca0d4","#dbd26b",
    "#807a40","#e04cfa","#d68fe2","#9932cc","#e6ccff","#02d659","#ad5100","#000000","#000000","#000000","#000000",
    "#000000","#000000","#CC66FF","#FF6666","#006400","#8d9e8b","#f5d5d5","#ff69b4","#ebf8b5","#000000","#000000",
    "#91ff36","#7dc975","#e97a7a","#0fffe3"]

vis_params = {
    'min': 0,
    'max': 69,
    'palette': palette,
    'format': "png"

}

# --- ADICIONA CADA ANO AO MAPA ---
for ano in anos_visualizar:
    banda = colecao_corrigida.select(f'classification_{ano}')
    Map.addLayer(img.select(f'classification_{ano}'), vis_params, f'Ano-original {ano}')
    Map.addLayer(banda, vis_params, f'Ano {ano}')

# --- EXIBE O MAPA INTERATIVO ---
Map.addLayerControl()
Map